In [0]:
%run ./00_setup_config

path,name,size,modificationTime
abfss://landing@sthealthcareamit.dfs.core.windows.net/appointments.csv,appointments.csv,10907,1783767831000
abfss://landing@sthealthcareamit.dfs.core.windows.net/billing.csv,billing.csv,10018,1783767831000
abfss://landing@sthealthcareamit.dfs.core.windows.net/doctors.csv,doctors.csv,962,1783767831000
abfss://landing@sthealthcareamit.dfs.core.windows.net/patients.csv,patients.csv,5626,1783767831000
abfss://landing@sthealthcareamit.dfs.core.windows.net/treatments.csv,treatments.csv,11072,1783767831000


Batch ID : c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff
Pipeline Version : 1.0
Run Time : 2026-07-11 14:04:34.997966


###Setting metadata

In [0]:
from pyspark.sql import functions as F

source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


batch_id,source_name,layer,rows_read,rows_written,status,start_time,end_time,error_message


In [0]:
billing_df = (
    spark.read
    .format("delta")
    .load(f"{bronze_path}billing")
)

display(billing_df.limit(20))

bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_pipeline_version,_ingestion_date
B001,P034,T001,2023-08-09,3941.97,Insurance,Pending,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B002,P032,T002,2023-06-09,4158.44,Insurance,Paid,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B003,P048,T003,2023-06-28,3731.55,Insurance,Paid,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B004,P025,T004,2023-09-01,4799.86,Insurance,Failed,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B005,P040,T005,2023-07-06,582.05,Credit Card,Pending,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B006,P045,T006,2023-06-19,1381.0,Insurance,Pending,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B007,P001,T007,2023-04-09,534.03,Cash,Failed,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B008,P016,T008,2023-05-24,3413.64,Cash,Failed,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B009,P039,T009,2023-03-05,4541.14,Credit Card,Paid,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11
B010,P005,T010,2023-01-13,1595.67,Cash,Paid,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11


In [0]:
patients_df = (
    spark.read
    .format("delta")
    .load(f"{silver_path}patients")
)

In [0]:
billing_df.printSchema()

print("Total Records :", billing_df.count())

display(billing_df.describe())


root
 |-- bill_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- treatment_id: string (nullable = true)
 |-- bill_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file_name: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _layer: string (nullable = true)
 |-- _pipeline_version: string (nullable = true)
 |-- _ingestion_date: date (nullable = true)

Total Records : 200


summary,bill_id,patient_id,treatment_id,amount,payment_method,payment_status,_source_file_name,_batch_id,_layer,_pipeline_version
count,200,200,200,200,200,200,200,200,200,200
mean,null,null,null,2756.2492500000003,null,null,null,null,null,1.0
stddev,null,null,null,1298.1253076474206,null,null,null,null,null,0.0
min,B001,P001,T001,534.03,Cash,Failed,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0
max,B200,P050,T200,4973.63,Insurance,Pending,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0


###Remove Duplicate Bills

In [0]:
billing_df = billing_df.dropDuplicates(["bill_id"])

In [0]:
print("After removing duplicates :", billing_df.count())

After removing duplicates : 200


###Remove Invalid Records

In [0]:
billing_df = billing_df.na.drop(
    subset=[
        "bill_id",
        "patient_id",
        "treatment_id"
    ]
)

In [0]:
billing_df = (

    billing_df

    .withColumn(
        "payment_method",
        F.upper(F.trim("payment_method"))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim("payment_status"))
    )

)

In [0]:
billing_df = billing_df.filter(
    F.col("amount") >= 0
)

In [0]:
billing_df = billing_df.filter(
    F.col("bill_date").isNotNull()
)

In [0]:
billing_df = billing_df.join(
    patients_df.select("patient_id"),
    on="patient_id",
    how="inner"
)

In [0]:
billing_df = billing_df.withColumn(
    "payment_completed",
    F.when(
        F.col("payment_status")=="PAID",
        1
    ).otherwise(0)
)

In [0]:
billing_df = billing_df.withColumn(
    "high_value_bill",
    F.when(
        F.col("amount")>=5000,
        1
    ).otherwise(0)
)

In [0]:
billing_df = (
    billing_df
    .withColumn(
        "_dq_passed",
        F.lit(True)
    )
    .withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(batch_id)
    )
    .withColumn(
        "_bronze_batch_id",
        F.col("_batch_id")
    )
    .withColumn(
        "_is_current",
        F.lit(True)
    )
    .withColumn(
        "_record_version",
        F.lit(1)
    )
)

In [0]:
billing_df.write \
.format("delta") \
.mode("overwrite") \
.save(f"{silver_path}billing")

In [0]:
silver_billing = (
    spark.read
    .format("delta")
    .load(f"{silver_path}billing")
)
display(silver_billing.limit(10))

patient_id,bill_id,treatment_id,bill_date,amount,payment_method,payment_status,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_pipeline_version,_ingestion_date,payment_completed,high_value_bill,_dq_passed,_silver_load_timestamp,_silver_batch_id,_bronze_batch_id,_is_current,_record_version
P031,B080,T080,2023-06-26,2426.9,CREDIT CARD,PENDING,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,0,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P035,B111,T111,2023-05-22,3787.93,CREDIT CARD,PAID,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,1,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P036,B104,T104,2023-04-18,2898.31,CREDIT CARD,FAILED,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,0,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P014,B147,T147,2023-11-13,4716.31,INSURANCE,FAILED,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,0,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P009,B185,T185,2023-03-21,1158.68,CASH,PENDING,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,0,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P008,B088,T088,2023-05-02,1733.72,CASH,PAID,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,1,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P023,B119,T119,2023-12-18,2911.22,CREDIT CARD,FAILED,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,0,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P050,B083,T083,2023-11-07,4960.65,CREDIT CARD,PENDING,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,0,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P032,B002,T002,2023-06-09,4158.44,INSURANCE,PAID,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,1,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
P047,B150,T150,2023-08-16,2286.42,CREDIT CARD,PAID,2026-07-11T11:51:45.445Z,billing.csv,dd177b19-f197-444a-9d50-bcf4524563e3,BRONZE,1.0,2026-07-11,1,0,true,2026-07-11T14:11:03.762Z,c4afd0ee-10d2-4be6-af7d-5d86a8acb4ff,dd177b19-f197-444a-9d50-bcf4524563e3,true,1
